Import Libraries

In [28]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


 
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

Data Processing

In [29]:
root ="news-dataset/news-dataset"

lang_codes =  {"english": "eng", "isixhosa": "xho", "chishona": "sna"}

def load_language(lang_code, data_root=root):
    #Load train/dev/test tsv files for one lang
 
    splits = {}
    for split in ["train", "dev", "test"]:
        path = os.path.join(data_root, lang_code, f"{split}.tsv")
        df = pd.read_csv(path, sep="\t")
        df["full_text"] = df["headline"].astype(str) + " " + df["text"].astype(str)
#Builds a full text column from headline and text

        splits[split] = df
        
    return splits



def get_label_encoder(train_df, label_col="category"):
    le = LabelEncoder()
    le.fit(train_df[label_col])
    return le



In [30]:
# test for english load lang
eng = load_language("eng")
for split, df in eng.items():
    print(split, df.shape)
eng["train"].head()

train (3309, 5)
dev (472, 5)
test (948, 5)


,category,headline,text,url,full_text
0,business,'We haven't had a single penny from the Post O...,Baljit Sethi cannot understand why it is takin...,/news/business-63889700,'We haven't had a single penny from the Post O...
1,entertainment,Redcar Regent Cinema: New venue to open on Fri...,kets have gone on sale for a new cinema which ...,/news/uk-england-tees-63260831,Redcar Regent Cinema: New venue to open on Fri...
2,health,Dorset County Hospital stands down critical in...,A main hospital has stood down its critical in...,/news/uk-england-dorset-64130718,Dorset County Hospital stands down critical in...
3,health,Watch: On the picket line with nurses across t...,"Nurses in Northern Ireland, Wales and England ...",/news/uk-64041760,Watch: On the picket line with nurses across t...
4,business,Sri Lanka urges farmers to plant more rice ami...,Sri Lanka is calling on farmers to grow more r...,/news/business-61655317,Sri Lanka urges farmers to plant more rice ami...


In [31]:
#test label encoder
eng_le = get_label_encoder(eng["train"])
print(eng_le.classes_)
print(eng_le.transform(["business", "sports", "technology"]))

['business' 'entertainment' 'health' 'politics' 'sports' 'technology']
[0 4 5]


Text cleaning and vectorisation

In [32]:
#Cleaning

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   
    # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()  
    # strip whitespace
    return text

def add_clean_column(df):
    df = df.copy()
    df["clean_text"] = df["full_text"].apply(clean_text)
    return df

In [33]:
#Vectorisation 


def build_vectoriser(method="tfidf", max_features=None, min_freq=1):
    if method == "cvec":
        return CountVectorizer(max_features=max_features, min_df=min_freq)
    elif method == "tfidf":
        return TfidfVectorizer(max_features=max_features, min_df=min_freq)
    else:
        raise ValueError(f"Unknown method: {method}")



def vectorise_splits(splits, method="tfidf", max_features=5000, min_freq=1):
    
    for name in splits:
        splits[name] = add_clean_column(splits[name])

    # make vectoriser
    vectoriser = build_vectoriser(method, max_features, min_freq)


    # choose which words make it and which dont
    x_train = vectoriser.fit_transform(splits["train"]["clean_text"])

    x_dev = vectoriser.transform(splits["dev"]["clean_text"])
    x_test = vectoriser.transform(splits["test"]["clean_text"])

    return (x_train, x_dev, x_test), vectoriser


In [34]:
#Tensors and data loaders 

def prepare_tensors(splits, vectoriser_output, label_col = "category"):
    (x_train, x_dev, x_test), vectorizer = vectoriser_output
    le = get_label_encoder(splits["train"], label_col)

    y_train = le.transform(splits["train"][label_col])
    y_dev   = le.transform(splits["dev"][label_col])
    y_test  = le.transform(splits["test"][label_col])

    return {
        "x_train": x_train, "y_train": y_train,
        "x_dev": x_dev,     "y_dev": y_dev,
        "x_test": x_test,   "y_test": y_test,
         "label_encoder": le,
    }


def make_dataloader(x,y, batch_size = 32, shuffle = False):
    x_dense = torch.tensor(x.toarray(), dtype = torch.float32)
    y_tensor = torch.tensor (y, dtype = torch.long)

    dataset = TensorDataset(x_dense, y_tensor)


    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

def build_dataloaders(tensors, batch_size= 32):
    train_loader = make_dataloader(tensors["x_train"], tensors["y_train"], batch_size, shuffle =True)
    dev_loader = make_dataloader(tensors["x_dev"], tensors["y_dev"], batch_size, shuffle =True)
    test_loader = make_dataloader(tensors["x_test"], tensors["y_test"], batch_size, shuffle =True)

    return train_loader, dev_loader, test_loader
    



In [35]:
#test vectorise splits, prep tensor and dataloader

vec_output = vectorise_splits(eng, method="tfidf", max_features=5000, min_freq=1)
tensors = prepare_tensors(eng, vec_output)
train_loader, dev_loader, test_loader = build_dataloaders(tensors, batch_size=32)

x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, x_batch.dtype)
print(y_batch.shape, y_batch.dtype)

torch.Size([32, 5000]) torch.float32
torch.Size([32]) torch.int64


Multinominal Logistic Regression

In [36]:
class MultinomialLogisticRegression(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_size, num_classes)


    def forward(self,x):
        return self.linear(x)


    def compute_probabilites(self, x):
        logits = self.forward(x)
        return F.softmax(logits, dim=1)

    def predict(self, x):
        prob = self.compute_probabilites(x)
        return torch.argmax(prob, dim= 1)

In [37]:
#testing  MultiNomialLogReg

test_model= MultinomialLogisticRegression(input_size= x_batch.shape[1], num_classes= len(tensors["label_encoder"].classes_))

logits =  test_model (x_batch)
prob = test_model.compute_probabilites(x_batch)
predictions = test_model.predict(x_batch)


print (logits.shape, prob.shape, predictions.shape)
print (prob[0])
print(prob[0].sum())
print(predictions[:5])


torch.Size([32, 6]) torch.Size([32, 6]) torch.Size([32])
tensor([0.1649, 0.1694, 0.1662, 0.1650, 0.1700, 0.1645],
       grad_fn=<SelectBackward0>)
tensor(1., grad_fn=<SumBackward0>)
tensor([4, 5, 1, 1, 1])


Training and early stopping

In [38]:
def train_model (model, train_loader , dev_loader, lr=0.01, num_epochs=20, patience=3):
    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    Loss = nn.CrossEntropyLoss()


    history = {"train_loss": [], 
               "Dev_loss":[],
               "Dev_acc":[] }

    best_acc =0
    best_model =None
    no_improvement =0

#train
    for epoch in range(num_epochs):

        model.train()
        train_loss= 0

        for x,y in train_loader:
            optimiser.zero_grad()


            output = model(x)
            loss = Loss(output,y)


            loss.backward()
            optimiser.step()


            train_loss = train_loss + loss.item()*x.size(0)
        train_loss = train_loss/ len(train_loader.dataset)

#validate
        
        model.eval()
        dev_loss =0
        correct = 0

        with torch.no_grad():

            for x,y in dev_loader:
                output = model(x)
                loss = Loss(output,y)

                dev_loss = dev_loss + loss.item()*x.size(0)
                correct = correct + (output.argmax(1)==y).sum().item()

        dev_loss = dev_loss / len(dev_loader.dataset)
        dev_acc = correct/ len(dev_loader.dataset)


        history["train_loss"].append(train_loss)
        history["Dev_loss"].append(dev_loss)
     
        history["Dev_acc"].append(dev_acc)


        print (f"Epoch {epoch+1}/{num_epochs}", f"Train loss:{train_loss:.4f}", f"Dev loss: {dev_loss:.4f}",f"Dev Accuracy:{dev_acc:.4f}")


        #early stop

        if  dev_acc>best_acc:
            best_acc= dev_acc
            best_model = model.state_dict().copy()
            no_improvement = 0
        else:
            no_improvement = no_improvement+1

        if no_improvement>= patience:
            print ("early stop")
            break

    if best_model is not None:
        model.load_state_dict(best_model
                              )


    return history









In [41]:
#test train model
test_model = MultinomialLogisticRegression(input_size=x_batch.shape[1], num_classes=len(tensors["label_encoder"].classes_))

history = train_model(test_model, train_loader, dev_loader, lr=0.1, num_epochs=50, patience = 3)

Epoch 1/50 Train loss:1.7644 Dev loss: 1.7373 Dev Accuracy:0.2500
Epoch 2/50 Train loss:1.7170 Dev loss: 1.6932 Dev Accuracy:0.2945
Epoch 3/50 Train loss:1.6736 Dev loss: 1.6514 Dev Accuracy:0.4449
Epoch 4/50 Train loss:1.6324 Dev loss: 1.6109 Dev Accuracy:0.5254
Epoch 5/50 Train loss:1.5926 Dev loss: 1.5721 Dev Accuracy:0.5763
Epoch 6/50 Train loss:1.5539 Dev loss: 1.5351 Dev Accuracy:0.5784
Epoch 7/50 Train loss:1.5172 Dev loss: 1.4997 Dev Accuracy:0.6970
Epoch 8/50 Train loss:1.4824 Dev loss: 1.4657 Dev Accuracy:0.6589
Epoch 9/50 Train loss:1.4486 Dev loss: 1.4329 Dev Accuracy:0.7182
Epoch 10/50 Train loss:1.4164 Dev loss: 1.4018 Dev Accuracy:0.7585
Epoch 11/50 Train loss:1.3853 Dev loss: 1.3724 Dev Accuracy:0.7691
Epoch 12/50 Train loss:1.3561 Dev loss: 1.3438 Dev Accuracy:0.7860
Epoch 13/50 Train loss:1.3279 Dev loss: 1.3164 Dev Accuracy:0.7903
Epoch 14/50 Train loss:1.3006 Dev loss: 1.2904 Dev Accuracy:0.8008
Epoch 15/50 Train loss:1.2744 Dev loss: 1.2658 Dev Accuracy:0.8157
Epoc